# Data Labeling and Preprocessing

Notebook ini digunakan untuk melakukan pelabelan sentimen berdasarkan skor ulasan, kemudian dilanjutkan dengan tahap preprocessing teks untuk mempersiapkan data sebelum pemodelan.

In [ ]:
pip install indoNLP

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 1.8 MB/s eta 0:00:00


In [ ]:
pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 2.8 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from indoNLP.preprocessing import replace_slang, replace_word_elongation
nltk.download('stopwords')
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Data Labeling

### Load Data

In [ ]:
df = pd.read_csv('/content/dana_reviews.csv')
print(f"Jumlah data: {len(df)}")
print("\n5 data pertama:")
df.head()

Jumlah data: 15000

5 data pertama:


,reviewId,content,score
0,d8621cfd-d005-4f9e-ab9b-e37273949401,good,5
1,8a622047-40e1-42e8-8a61-a5da313bab5b,dana sangat bagus,4
2,c144ae14-09fb-4d53-bd4e-e9bd1310cea5,mantap sangat membantu financial,5
3,262eb4a6-0f84-4c38-8f2a-4fabdae93c8d,sangat baik,5
4,262c3d49-429b-4910-bbff-df3cbb87979a,jozzz,4


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   reviewId  15000 non-null  object
 1   content   15000 non-null  object
 2   score     15000 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 351.7+ KB


Pelabelan Sentimen berdasarkan Skor, dengan skor 1-2 menunjukkan sentimen negative, skor 3 menunjukkan sentimen neutral, dan skor 4-5 menunjukkan sentimen positif

In [ ]:
df['label'] = ''

df.loc[df['score'] <= 2, 'label'] = 'negative'
df.loc[df['score'] == 3, 'label'] = 'neutral'
df.loc[df['score'] >= 4, 'label'] = 'positive'

In [ ]:
df.head()

,reviewId,content,score,label
0,d8621cfd-d005-4f9e-ab9b-e37273949401,good,5,positive
1,8a622047-40e1-42e8-8a61-a5da313bab5b,dana sangat bagus,4,positive
2,c144ae14-09fb-4d53-bd4e-e9bd1310cea5,mantap sangat membantu financial,5,positive
3,262eb4a6-0f84-4c38-8f2a-4fabdae93c8d,sangat baik,5,positive
4,262c3d49-429b-4910-bbff-df3cbb87979a,jozzz,4,positive


## Data Pre-Processing

In [ ]:
df_selected = df[['reviewId', 'content', 'label']]

In [ ]:
df_selected.head()

,reviewId,content,label
0,d8621cfd-d005-4f9e-ab9b-e37273949401,good,positive
1,8a622047-40e1-42e8-8a61-a5da313bab5b,dana sangat bagus,positive
2,c144ae14-09fb-4d53-bd4e-e9bd1310cea5,mantap sangat membantu financial,positive
3,262eb4a6-0f84-4c38-8f2a-4fabdae93c8d,sangat baik,positive
4,262c3d49-429b-4910-bbff-df3cbb87979a,jozzz,positive


In [ ]:
print(df_selected.isnull().sum())

reviewId    0
content     0
label       0
dtype: int64


### Case Folding

In [ ]:
df_selected['teks_lower'] = df_selected['content'].str.lower()

In [ ]:
df_selected[['content', 'teks_lower']].head()

,content,teks_lower
0,good,good
1,dana sangat bagus,dana sangat bagus
2,mantap sangat membantu financial,mantap sangat membantu financial
3,sangat baik,sangat baik
4,jozzz,jozzz


### Special Characters Removal

In [ ]:
def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
df_selected['teks_clean'] = df_selected['teks_lower'].apply(clean_text)

In [ ]:
df_selected[['teks_lower', 'teks_clean']].head()

,teks_lower,teks_clean
0,good,good
1,dana sangat bagus,dana sangat bagus
2,mantap sangat membantu financial,mantap sangat membantu financial
3,sangat baik,sangat baik
4,jozzz,jozzz


### Normalisasi

In [ ]:
def normalize_slang(text):
    text = replace_slang(text)
    text = replace_word_elongation(text)
    return text

df_selected['teks_normalized'] = df_selected['teks_clean'].apply(normalize_slang)

In [ ]:
df_selected[['teks_clean', 'teks_normalized']].head()

,teks_clean,teks_normalized
0,good,good
1,dana sangat bagus,dana sangat bagus
2,mantap sangat membantu financial,mantap sangat membantu financial
3,sangat baik,sangat baik
4,jozzz,joz


### Tokenization

In [ ]:
df_selected['tokens'] = df_selected['teks_normalized'].apply(lambda x: x.split())

In [ ]:
df_selected[['teks_normalized', 'tokens']].head()

,teks_normalized,tokens
0,good,[good]
1,dana sangat bagus,"[dana, sangat, bagus]"
2,mantap sangat membantu financial,"[mantap, sangat, membantu, financial]"
3,sangat baik,"[sangat, baik]"
4,joz,[joz]


### Stopword Removal

In [ ]:
def remove_stopwords(tokens):
    text = ' '.join(tokens)
    text = stopword_remover.remove(text)
    return text.split()

df_selected['tokens_final'] = df_selected['tokens'].apply(remove_stopwords)

In [ ]:
df_selected[['tokens', 'tokens_final']].head()

,tokens,tokens_final
0,[good],[good]
1,"[dana, sangat, bagus]","[dana, sangat, bagus]"
2,"[mantap, sangat, membantu, financial]","[mantap, sangat, membantu, financial]"
3,"[sangat, baik]","[sangat, baik]"
4,[joz],[joz]


### Stemming

In [1]:
def stemming_tokens(tokens):
    text = ' '.join(tokens)
    stemmed_text = stemmer.stem(text)
    return stemmed_text.split()

In [ ]:
df_selected['tokens_stemmed'] = df_selected['tokens_final'].apply(stemming_tokens)

In [ ]:
df_selected[['tokens_final', 'tokens_stemmed']].head()

,tokens_final,tokens_stemmed
0,[good],[good]
1,"[dana, sangat, bagus]","[dana, sangat, bagus]"
2,"[mantap, sangat, membantu, financial]","[mantap, sangat, bantu, financial]"
3,"[sangat, baik]","[sangat, baik]"
4,[joz],[joz]


### Join Tokens

In [ ]:
def join_tokens(tokens):
  return ' '.join(tokens)

In [ ]:
df_selected['teks_stemmed'] = df_selected['tokens_stemmed'].apply(join_tokens)

In [ ]:
df_selected[['tokens_stemmed', 'teks_stemmed']].head()

,tokens_stemmed,teks_stemmed
0,[good],good
1,"[dana, sangat, bagus]",dana sangat bagus
2,"[mantap, sangat, bantu, financial]",mantap sangat bantu financial
3,"[sangat, baik]",sangat baik
4,[joz],joz


In [ ]:
print(df_selected.isnull().sum())

reviewId           0
content            0
label              0
teks_lower         0
teks_clean         0
teks_normalized    0
tokens             0
tokens_final       0
tokens_stemmed     0
teks_stemmed       0
dtype: int64


# Save Data

In [ ]:
df_selected.to_csv("dana_preprocessing.csv", index=False)